<img src="https://theaiengineer.dev/tae_logo_gw_flatter.png" width="35%" align="right">


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yhilpisch/colab/blob/main/notebooks/03_gpu_dataframes_cudf.ipynb)


# colab — GPU-Accelerated DataFrames with cuDF

## _Pandas-like Data Science at GPU Speed on a Free T4_

**&copy; Dr. Yves J. Hilpisch**<br>AI-Powered by Different LLMs.

## How to Use This Notebook

- **Goal**: Show how RAPIDS `cuDF` turns pandas workflows into GPU-accelerated
  operations with zero code changes for many common patterns.
- **Hardware**: Designed for Google Colab (T4 default). GPU is auto-detected.
- **Data**: Purely synthetic — no uploads or external downloads required.
- **GPU Reference**: See `notebooks/colab_gpus.md` for the full GPU comparison table.

### Roadmap

1. **Environment**: Check GPU and install `cudf-cu12`.
2. **Synthetic Data**: Generate a 5 M row dataframe with numeric and string columns.
3. **GroupBy**: Compare `groupby().agg()` on CPU pandas vs GPU cuDF.
4. **Merge**: Compare a join of two large tables.
5. **Rolling**: Compute a rolling window mean.
6. **String ops**: Filter rows with `str.contains()`.
7. **Summary**: Side-by-side timing table and speed-up bar chart.

### Environment Setup

We verify the GPU and install the RAPIDS cuDF library.

In [ ]:
#@title Check GPU and install RAPIDS cuDF
!nvidia-smi
!pip -q install cudf-cu12 --extra-index-url=https://pypi.nvidia.com
!pip -q install matplotlib

In [ ]:
#@title Imports and timing helper
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import cudf

def bench(label, fn, *args, **kwargs):
    start = time.perf_counter()
    result = fn(*args, **kwargs)
    elapsed = time.perf_counter() - start
    print(f"{label}: {elapsed:.3f} s")
    return result, elapsed

### Synthetic Data Generation

We create a 5 M row dataframe with numeric columns, a categorical column,
and a string column — enough to make CPU pandas feel the strain.

In [ ]:
#@title Generate 5M row synthetic dataset
N_ROWS = 5_000_000
np.random.seed(42)

df_cpu = pd.DataFrame({
    "id": np.arange(N_ROWS),
    "group": np.random.choice(["A", "B", "C", "D", "E"], size=N_ROWS),
    "value_1": np.random.randn(N_ROWS),
    "value_2": np.random.randn(N_ROWS) * 100,
    "value_3": np.random.randint(0, 1000, size=N_ROWS),
})

# String column for text filtering
words = ["alpha", "beta", "gamma", "delta", "epsilon"]
df_cpu["text"] = np.random.choice(words, size=N_ROWS)

df_gpu = cudf.from_pandas(df_cpu)
print(f"CPU dataframe: {df_cpu.shape}")
print(f"GPU dataframe: {df_gpu.shape}")

### Example 1: GroupBy Aggregation

Group by a categorical column and compute mean, std, and count.
This is one of the most common data-wrangling patterns.

In [ ]:
#@title CPU pandas groupby
_, t_cpu_gb = bench(
    "CPU groupby",
    lambda: df_cpu.groupby("group")["value_1"].agg(["mean", "std", "count"]),
)

In [ ]:
#@title GPU cuDF groupby
_, t_gpu_gb = bench(
    "GPU groupby",
    lambda: df_gpu.groupby("group")["value_1"].agg(["mean", "std", "count"]),
)

speedup_gb = t_cpu_gb / t_gpu_gb
print(f"\n>>> GroupBy speed-up: {speedup_gb:.1f}×")

### Example 2: Merge (Join)

Join the 5 M row table with a 50 k row lookup table on the `group` key.
Joins are expensive on CPU but map well to GPU parallel hash joins.

In [ ]:
#@title Prepare lookup table
lookup_cpu = pd.DataFrame({
    "group": ["A", "B", "C", "D", "E"],
    "category": ["cat1", "cat2", "cat3", "cat4", "cat5"],
    "multiplier": [1.0, 1.5, 2.0, 2.5, 3.0],
})
lookup_gpu = cudf.from_pandas(lookup_cpu)
print(lookup_cpu)

In [ ]:
#@title CPU merge
_, t_cpu_mg = bench(
    "CPU merge",
    lambda: df_cpu.merge(lookup_cpu, on="group", how="left"),
)

In [ ]:
#@title GPU merge
_, t_gpu_mg = bench(
    "GPU merge",
    lambda: df_gpu.merge(lookup_gpu, on="group", how="left"),
)

speedup_mg = t_cpu_mg / t_gpu_mg
print(f"\n>>> Merge speed-up: {speedup_mg:.1f}×")

### Example 3: Rolling Window Mean

Compute a 100-point rolling mean over a sorted numeric column.
Rolling window operations are memory-bound and benefit strongly from GPU bandwidth.

In [ ]:
#@title CPU rolling mean
_, t_cpu_rl = bench(
    "CPU rolling",
    lambda: df_cpu.sort_values("id")["value_2"].rolling(100).mean(),
)

In [ ]:
#@title GPU rolling mean
_, t_gpu_rl = bench(
    "GPU rolling",
    lambda: df_gpu.sort_values("id")["value_2"].rolling(100).mean(),
)

speedup_rl = t_cpu_rl / t_gpu_rl
print(f"\n>>> Rolling speed-up: {speedup_rl:.1f}×")

### Example 4: String Filtering

Filter rows where a string column contains a substring.
String operations are traditionally slow on CPUs; cuDF accelerates them via GPU
kernels.


In [ ]:
#@title CPU string contains
_, t_cpu_str = bench(
    "CPU string filter",
    lambda: df_cpu[df_cpu["text"].str.contains("alpha|beta")],
)

In [ ]:
#@title GPU string contains
_, t_gpu_str = bench(
    "GPU string filter",
    lambda: df_gpu[df_gpu["text"].str.contains("alpha|beta")],
)

speedup_str = t_cpu_str / t_gpu_str
print(f"\n>>> String filter speed-up: {speedup_str:.1f}×")

## Summary: Speed-Up Comparison

Even a modest free T4 GPU can deliver 5–50× speed-ups on common dataframe operations.

In [ ]:
#@title Build and display results
results = pd.DataFrame({
    "Workload": [
        "GroupBy (5M rows)",
        "Merge (5M x 5)",
        "Rolling mean (5M)",
        "String filter (5M)",
    ],
    "CPU time (s)": [
        round(t_cpu_gb, 2),
        round(t_cpu_mg, 2),
        round(t_cpu_rl, 2),
        round(t_cpu_str, 2),
    ],
    "GPU time (s)": [
        round(t_gpu_gb, 2),
        round(t_gpu_mg, 2),
        round(t_gpu_rl, 2),
        round(t_gpu_str, 2),
    ],
})
results["Speed-up"] = results["CPU time (s)"] / results["GPU time (s)"]
results["Speed-up"] = (
    results["Speed-up"].round(1).astype(str) + "×"
)

print(results.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(results))
width = 0.35
ax.bar(
    x - width / 2,
    results["CPU time (s)"],
    width,
    label="CPU",
    color="#e74c3c",
)
ax.bar(
    x + width / 2,
    results["GPU time (s)"],
    width,
    label="GPU",
    color="#2ecc71",
)
ax.set_ylabel("Wall time (seconds, log scale)")
ax.set_yscale("log")
ax.set_xticks(x)
ax.set_xticklabels(
    results["Workload"],
    rotation=15,
    ha="right",
)
ax.legend()
ax.set_title("CPU vs GPU wall time")
plt.tight_layout()
plt.show()

### Exercises

1. **Scale the data**: Increase `N_ROWS` to 20 M. Does the speed-up grow or shrink?
2. **More columns**: Add 10 additional numeric columns and re-run the groupby.
   Is the speed-up consistent?
3. **Complex aggregation**: Replace the simple groupby with `.agg({...})`
   applying different functions to different columns.
4. **Real data**: Upload a large CSV (e.g. from Kaggle) and rerun the benchmarks.
5. **Dask + cuDF**: Explore `dask_cudf` for out-of-core GPU dataframe processing.

<img src="https://theaiengineer.dev/tae_logo_gw_flatter.png" width="35%" align="right">
